# Inference Notebook: `inference_example.py`

This notebook explains what the deployment inference code does. Inference means: use the already-trained model to make a prediction.

## Big Picture

`artifacts/v1.0.0/inference_example.py` is the small reference version of production prediction.

It does this:

1. Load model metadata from `model_card.json`.
2. Load tokenizer.
3. Load ONNX model.
4. Load LogisticRegression classifier.
5. Build text from invoice fields.
6. Turn text into an embedding.
7. Get category probabilities.
8. Return top-3 predictions and review/auto-accept decision.

## Beginner Python Words

- `class Classifier`: creates a reusable predictor object.
- `__init__`: runs once when the object is created.
- `self`: means “this object.” It stores things the object remembers.
- `Path`: handles file paths.
- `json.loads`: turns JSON text into Python dictionaries/lists.
- `np.ndarray`: a NumPy array, used for model numbers.
- `if`: choose one path or another.
- `for`: repeat work for each item.

In [ ]:
from pathlib import Path
import json

ARTIFACT_DIR = Path('../artifacts/v1.0.0')
card = json.loads((ARTIFACT_DIR / 'model_card.json').read_text())

card['model_version'], card['artifact_format'], card['trained_classes']

## Loading The Pieces

The deployed package has several files:

- `model.onnx`: transformer body, used to make embeddings.
- `tokenizer/`: turns text into token numbers.
- `classifier.joblib`: LogisticRegression head, used to pick category scores.
- `labels.json`: names, weak classes, trained/untrained classes.
- `model_card.json`: version, metrics, thresholds.

## Building Input Text

The model expects one string. The code joins fields like this:

`item_text | description | provider`

If description is only numbers, it gets removed because product codes usually do not help meaning.

In [ ]:
import re

NUMERIC_RE = re.compile(r'[\d\s.,\-/]+$')

def build_text(item_text, description='', provider=''):
    description = (description or '').strip()
    if NUMERIC_RE.fullmatch(description or '0'):
        description = ''
    parts = [item_text.strip(), description, (provider or '').strip()]
    return ' | '.join(part for part in parts if part)

build_text('VACUNA CLOSTRIBAC 8 GOLD X 50 DOS.', '10000026', 'COOPRINSEM')

## Tokenizer → ONNX → Embedding

The tokenizer converts words into numbers. The ONNX model converts those numbers into many hidden vectors. Then the code mean-pools them into one embedding.

Mean pooling means: average the token vectors into one vector for the whole line item.

## Classifier Scores

`predict_proba` returns one score per trained class. The code sorts those scores and keeps the top 3.

Top-3 is important because the UI can show three choices to the human reviewer.

## Decision Logic

The decision is simple:

- weak class → review
- top score below threshold → review
- first and second too close → review
- otherwise → auto-accept

This keeps the model useful without pretending it is perfect.

In [ ]:
# Optional: run this only in an environment with the model runtime installed.

"""
import sys
sys.path.insert(0, str(ARTIFACT_DIR))
from inference_example import Classifier

clf = Classifier(ARTIFACT_DIR)
clf.predict('VACUNA CLOSTRIBAC 8 GOLD X 50 DOS.', provider='COOPRINSEM')
"""

## What Changed In The Backend

The FastAPI backend uses the same idea, but splits the work into smaller files:

- encoder file: ONNX embedding
- classifier file: probability scores
- predictor file: full prediction flow
- routes file: web API endpoints

Same model logic, cleaner web service shape.